In [1]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt


class BeginnerGridWorld:

    def __init__(self):
        self.height = 10
        self.width = 10
        self.num_states = 100
        self.num_actions = 4

        self.goal = 99
        self.trap = 55

        self.P = np.zeros((100, 4, 100))
        self.R = np.zeros((100, 4))

        self.build_world()

    def build_world(self):
        for s in range(self.num_states):

            if s == self.goal or s == self.trap:
                for a in range(self.num_actions):
                    self.P[s, a, s] = 1.0
                continue

            r, c = s // 10, s % 10

            neighbors = {
                0: (max(r - 1, 0), c),
                1: (min(r + 1, 9), c),
                2: (r, max(c - 1, 0)),
                3: (r, min(c + 1, 9))
            }

            for a in range(self.num_actions):

                intended_state = neighbors[a][0] * 10 + neighbors[a][1]
                self.P[s, a, intended_state] += 0.85

                for slip_action in range(self.num_actions):
                    if slip_action != a:
                        slip_state = neighbors[slip_action][0] * 10 + neighbors[slip_action][1]
                        self.P[s, a, slip_state] += 0.05

                for s_next in range(self.num_states):
                    prob = self.P[s, a, s_next]

                    if prob > 0:
                        if s_next == self.goal:
                            self.R[s, a] += prob * 10.0
                        elif s_next == self.trap:
                            self.R[s, a] += prob * -10.0
                        else:
                            self.R[s, a] += prob * -0.1

    def solve_value_iteration(self, gamma=0.99, theta=1e-8, max_iters=1000):
        V = np.zeros(self.num_states)

        for i in range(max_iters):

            Q = self.R + gamma * np.tensordot(self.P, V, axes=(2, 0))

            V_new = np.max(Q, axis=1)

            delta = np.max(np.abs(V_new - V))
            V = V_new

            if delta < theta:
                print(f"Value Iteration converged in {i + 1} iterations!")
                break

        final_Q = self.R + gamma * np.tensordot(self.P, V, axes=(2, 0))
        optimal_policy = np.argmax(final_Q, axis=1)

        return V, optimal_policy

    def solve_policy_iteration(self, gamma=0.99, max_iters=100):
        policy = np.zeros(self.num_states, dtype=int)
        V = np.zeros(self.num_states)

        state_indices = np.arange(self.num_states)

        for i in range(max_iters):

            P_pi = self.P[state_indices, policy, :]
            R_pi = self.R[state_indices, policy]

            A = np.eye(self.num_states) - gamma * P_pi

            V = scipy.linalg.solve(A, R_pi)

            Q = self.R + gamma * np.tensordot(self.P, V, axes=(2, 0))
            new_policy = np.argmax(Q, axis=1)

            if np.array_equal(new_policy, policy):
                print(f"Policy Iteration converged in {i + 1} steps!")
                break

            policy = new_policy

        return V, policy


if __name__ == "__main__":

    env = BeginnerGridWorld()

    discount_factors = [0.9, 0.95, 0.99, 0.999]

    for gamma in discount_factors:
        print(f"\n--- Running Dynamic Programming for Gamma = {gamma} ---")

        v_vi, pi_vi = env.solve_value_iteration(gamma=gamma)
        v_pi, pi_pi = env.solve_policy_iteration(gamma=gamma)

        max_diff = np.max(np.abs(v_vi - v_pi))
        print(f"Max Value discrepancy between VI and PI: {max_diff:.2e}")

    optimal_values, optimal_policy = env.solve_policy_iteration(gamma=0.99)

    plt.figure(figsize=(8, 6))
    plt.imshow(optimal_values.reshape((10, 10)), cmap="viridis", origin="upper")
    plt.colorbar(label="Expected Value $V(s)$")
    plt.title(r"Dynamic Programming: Optimal Value Grid ($\gamma=0.99$)")

    arrows = {
        0: "↑",
        1: "↓",
        2: "←",
        3: "→"
    }

    for r in range(10):
        for c in range(10):
            idx = r * 10 + c

            if idx == env.goal:
                label = "G"
            elif idx == env.trap:
                label = "T"
            else:
                label = arrows[optimal_policy[idx]]

            plt.text(
                c,
                r,
                label,
                ha="center",
                va="center",
                color="white" if optimal_values[idx] < 0 else "black",
                fontweight="bold"
            )

    plt.savefig("mdp_optimal_values.png")
    plt.close()

    print("\nState Values visual plot saved as 'mdp_optimal_values.png'.")


--- Running Dynamic Programming for Gamma = 0.9 ---
Value Iteration converged in 52 iterations!
Max Value discrepancy between VI and PI: 8.68e-09

--- Running Dynamic Programming for Gamma = 0.95 ---
Value Iteration converged in 57 iterations!
Max Value discrepancy between VI and PI: 8.67e-09

--- Running Dynamic Programming for Gamma = 0.99 ---
Value Iteration converged in 61 iterations!
Policy Iteration converged in 14 steps!
Max Value discrepancy between VI and PI: 1.19e-08

--- Running Dynamic Programming for Gamma = 0.999 ---
Value Iteration converged in 62 iterations!
Max Value discrepancy between VI and PI: 1.20e-08
Policy Iteration converged in 14 steps!

State Values visual plot saved as 'mdp_optimal_values.png'.
